# 1. How we measure, and where the open models start

Every later notebook compares systems, so the scoring is fixed first and never changed afterwards: one text
normalizer, corpus word error rate (WER), and 95% confidence intervals that resample whole recordings or speakers
(clips from one recording are not independent). The media-speech test set is cut from the DL Sprint 4.0 recordings by
forced alignment and split by recording, so no recording is in both training and test data.

In [1]:
from nbutil import *
import pandas as pd

## The test sets
Clips, recordings or speakers (the bootstrap clusters) and reference words, read from the scored results.

In [2]:
rows = {}
for s, name in SETS.items():
    m = load(ZERO_SHOT.format(s) + "/metrics.json")
    rows[name] = {"clips": m["n_utterances"], "clusters (recordings / speakers)": m["n_groups"], "reference words": m["ref_words"]}
pd.DataFrame(rows).T

,clips,clusters (recordings / speakers),reference words
DL Sprint eval,4234,22,120029
FLEURS,920,349,17766
Common Voice,9408,7377,86891
Kathbath,2236,17,23995


## Training data never touches the test sets
Checked for the final model's whole training list (`analysis/own_contamination.py`): recordings, speakers and sentences.

In [3]:
c = load("results/contamination/own_model.json")
display(pd.Series({k: v for k, v in c.items() if isinstance(v, int)}))
pd.DataFrame({s: {"clips": v["clips"], "clips sharing a training sentence": v["clips_with_training_sentence"],
                  "final-model WER": c["final_model_wer_without_overlapping_clips"][s]["all"],
                  "WER without those clips": c["final_model_wer_without_overlapping_clips"][s]["without"]}
              for s, v in c["sentence_overlap"].items()}).T

training_clips                                         96665
dlsprint_training_clips_from_dev_or_eval_recordings        0
cv_test_speakers_in_training                               0
kathbath_valid_speakers_in_training                        0
fleurs_clips_in_training                                   0
dtype: int64

,clips,clips sharing a training sentence,final-model WER,WER without those clips
splits_fa-eval,4234.0,6.0,11.176,11.174
fleurs-test,920.0,0.0,15.597,15.597
cv-test,9408.0,3.0,10.133,10.135
kathbath-valid,2236.0,0.0,10.281,10.281


## The normalizer and the WER
The exact code every number in this repository goes through:

In [4]:
from bangla_asr.common.text import normalize_bangla_text
from bangla_asr.common.stats import edit_counts, corpus_rates
source(normalize_bangla_text); source(edit_counts); source(corpus_rates)

def normalize_bangla_text(text: str) -> str:
    """
    Standardizes Bangla text for WER/CER evaluation and metric comparison.
    Recomposes decomposed nukta graphemes (ড়, ঢ়, য়), strips punctuation, and collapses whitespace.
    """
    if not text:
        return ""
    text = unicodedata.normalize("NFC", str(text))

    # Canonicalize decomposed nukta combinations into single unified Bengali code points
    text = text.replace("\u09a1\u09bc", "\u09dc")  # ড + ় -> ড়
    text = text.replace("\u09a2\u09bc", "\u09dd")  # ঢ + ় -> ঢ়
    text = text.replace("\u09af\u09bc", "\u09df")  # য + ় -> য়

    # Strip punctuation
    text = _PUNCT_REGEX.sub(" ", text)

    # Collapse multiple whitespaces
    text = _WHITESPACE_REGEX.sub(" ", text).strip()
    return text

def edit_counts(hyp: str, ref: str) -> Dict[str, int]:
    """Word and character edit counts of one utterance (normalized text)."""
    h, r = normalize_bangla_text(hyp), normalize_bangla_text(ref)
    hw, rw = h.split(), r.spl

## Open models without any training
WER (%) on each test set. The tugstugi Whisper-medium model is the base for everything that follows.

In [5]:
wer_table({
    "tugstugi Whisper-medium": ZERO_SHOT,
    "Hishab conformer large *": "results/zeroshot/{}/hishab__titu_stt_bn_conformer_large",
    "BanglaASR (Whisper-small)": "results/zeroshot/{}/bangla-speech-processing__BanglaASR",
    "IndicWav2Vec": "results/zeroshot/{}/ai4bharat__indicwav2vec_v1_bengali",
    "wav2vec2-xls-r-300m-bn": "results/zeroshot/{}/arijitx__wav2vec2-xls-r-300m-bengali",
})

,DL Sprint eval,FLEURS,Common Voice,Kathbath
tugstugi Whisper-medium,22.70,21.54,14.80,14.81
Hishab conformer large *,21.18,14.24,7.72,8.32
BanglaASR (Whisper-small),58.20,32.94,9.66,32.20
IndicWav2Vec,42.80,28.18,25.80,22.15
wav2vec2-xls-r-300m-bn,65.20,47.46,43.35,38.57


\* Hishab's model card lists Kathbath and Common Voice among its training data, so those two cells are not comparable.

In [6]:
metrics(ZERO_SHOT.format("splits_fa-eval"))

{'WER': 22.7, '95% CI': '[20.69, 25.32]', 'CER': 7.7, 'clips': 4234}

In [7]:
record("results/modal/zeroshot/splits_fa-eval/tugstugi_a100/run_info.json")

**Run record** `results/modal/zeroshot/splits_fa-eval/tugstugi_a100/run_info.json`  
`python /root/proj/pipeline/eval_zero_shot.py --manifest data/splits_fa/eval.csv --model bengaliAI/tugstugi_bengaliai-regional-asr_whisper-medium --name tugstugi_a100 --out_dir results/zeroshot --batch_size 8`  
commit `` · started 2026-10-09T22:43:31 · 1.0 s · torch 2.13.0+cu130, transformers ?, GPU NVIDIA A100-SXM4-40GB

**Next:** the base model makes 22.7% word errors on media speech. The cheapest way to improve a recognizer without
training is to let it propose several transcripts and choose among them (rescoring); notebook 2 tests whether that works.